# Demo orchestrator

In [ ]:
import sys
import os
import random
import torch
import pandas as pd
import logging

# 1. Setup path to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 2. Import project logger config
from logger import set_level

# 3. Import Pipeline components
from peptide_pipeline.orchestrator.orchestrator import Orchestrator
from peptide_pipeline.chemist.chemist_agent import ChemistAgent
from peptide_pipeline.biologist.esm_biologist_zscore import ESMBiologistZscore
from peptide_pipeline.generator.vae_generator import VAEGenerator

# 4. Configure logging to INFO to see agents' activity
set_level(logging.INFO)

## 1. Préparation et Entraînement du Générateur (VAE)

In [ ]:
SEQ_LENGTH = 6 
INPUT_DIM = SEQ_LENGTH * 20  # 120
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")

# --- 1.1 Génération de données synthétiques ---
print("Generating synthetic training data...")
random.seed(42)
train_peptides = list(set(
    ''.join(random.choices(AMINO_ACIDS, k=SEQ_LENGTH))
    for _ in range(1000)
))

# --- 1.2 Encodage One-Hot ---
aa_index = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
def encode(sequences, seq_length):
    tensor = torch.zeros(len(sequences), seq_length * 20)
    for i, seq in enumerate(sequences):
        for j, aa in enumerate(seq):
            if j < seq_length:
                tensor[i, j * 20 + aa_index[aa]] = 1.0
    return tensor

train_data = encode(train_peptides, SEQ_LENGTH)

# --- 1.3 Initialisation et Entraînement ---
generator = VAEGenerator(input_dim=INPUT_DIM, latent_dim=32, hidden_dim=128)
print(f"Device: {generator.device}")

print("Starting VAE warm-up training...")
generator.train_model(train_data.to(generator.device), epochs=150, batch_size=64, lr=1e-3)
print("VAE Ready!")

## 2. Initialisation des Agents & Orchestrateur

In [ ]:
chemist = ChemistAgent()

# Biologiste : On utilise 'RLLKRF' comme référence (donc on cherche des peptides similaires structurellement)
biologist = ESMBiologistZscore(
    reference_peptide="RLLKRF", 
    model_name="facebook/esm2_t6_8M_UR50D"
)

orchestrator = Orchestrator(generator, chemist, biologist)
print("Orchestrator Initialized.")

## 3. Exécution de la Pipeline

Nous partons du peptide **"RLLKRF"**. L'orchestrateur va :
1. Demander au VAE de générer des variantes (mutation via l'espace latent).
2. Filtrer chimiquement.
3. Scorer biologiquement.
4. Répéter le processus sur les survivants.

In [ ]:
results = orchestrator.run(
    nb_iterations=5,
    nb_peptides=50,
    top_k=5,
    exploration_rate=0.3, # Bruit injecté dans l'espace latent pour créer des variants
    initial_peptide="RLLKRF"
)

print(f"\nPipeline Finished. {len(results)} candidates found.")

## 4. Résultats

In [ ]:
df_results = []
for res in results:
    row = {"Peptide": res["peptide"], "Score ESM": res["score"]}
    row.update(res["properties"])
    df_results.append(row)

df = pd.DataFrame(df_results)
if not df.empty:
    cols = ['Peptide', 'Score ESM', 'molecular_weight', 'logp']
    cols += [c for c in df.columns if c not in cols]
    display(df[cols])
else:
    print("Aucun résultat.")